CFTC Positioning
- Main DataFrame

In [152]:
import pandas as pd
import numpy as np
import plotly.express as px
import streamlit as st
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

dfs = []

file_folder = "../data/cftc/historicals/"

for file_path in Path(file_folder).glob("cftc_*.txt"):
    temp_df = pd.read_csv(file_path, low_memory = False)
    dfs.append(temp_df)

cftc_df = pd.concat(dfs, ignore_index=True).copy()
# Create date fields
cftc_df["Date"] = pd.to_datetime(
    cftc_df["Report_Date_as_YYYY-MM-DD"],
    errors="coerce"
)

cftc_df["Year"] = cftc_df["Date"].dt.year
cftc_df["Month"] = cftc_df["Date"].dt.month

In [153]:


cftc_df = cftc_df.sort_values(
    "Report_Date_as_YYYY-MM-DD"
).reset_index(drop=True)

In [154]:
files = sorted(Path(file_folder).glob("cftc_*.txt"))

column_sets = {}

for file_path in files:
    df_test = pd.read_csv(file_path, low_memory=False)
    column_sets[file_path.name] = set(df_test.columns)

base_file = files[0].name
base_columns = column_sets[base_file]

for file_name, columns in column_sets.items():

    missing = base_columns - columns
    extra = columns - base_columns

    print(f"\n{file_name}")

    if missing:
        print("Missing:", missing)

    if extra:
        print("Extra:", extra)

    if not missing and not extra:
        print("Schema: OK")


cftc_2019.txt
Schema: OK

cftc_2020.txt
Schema: OK

cftc_2021.txt
Schema: OK

cftc_2022.txt
Schema: OK

cftc_2023.txt
Schema: OK

cftc_2024.txt
Schema: OK

cftc_2025.txt
Schema: OK

cftc_2026.txt
Schema: OK


In [155]:
corn = cftc_df[
    cftc_df["Market_and_Exchange_Names"].str.contains(
        "CORN",
        case=False,
        na=False
    )
].copy()

print(
    corn.groupby("Year")["Date"]
    .agg(["min", "max", "count"])
)

            min        max  count
Year                             
2019 2019-01-08 2019-12-31     59
2020 2020-01-07 2020-12-29     52
2021 2021-01-05 2021-12-28     52
2022 2022-01-04 2022-12-27     59
2023 2023-01-03 2023-12-26     86
2024 2024-01-02 2024-12-31     53
2025 2025-01-07 2025-12-30     87
2026 2026-01-06 2026-09-22     39


In [156]:
corn = cftc_df[
    cftc_df["Market_and_Exchange_Names"].str.contains(
        "CORN",
        case=False,
        na=False
    )
].copy()

print(
    corn["Market_and_Exchange_Names"]
    .value_counts()
)

Market_and_Exchange_Names
CORN - CHICAGO BOARD OF TRADE                    403
CORN CONSECUTIVE CSO - CHICAGO BOARD OF TRADE     51
CORN CSO - CHICAGO BOARD OF TRADE                 33
Name: count, dtype: int64


Rename Columns

In [157]:
market_categories = {
    "Grains": [
        "WHEAT-SRW - CHICAGO BOARD OF TRADE",
        "WHEAT-HRW - CHICAGO BOARD OF TRADE",
        "WHEAT-HRSpring - MIAX FUTURES EXCHANGE",
        "BLACK SEA WHEAT FINANCIAL - CHICAGO BOARD OF TRADE",
        "WHEAT-HRSpring - MINNEAPOLIS GRAIN EXCHANGE",
        "CORN - CHICAGO BOARD OF TRADE",
        "CORN CONSECUTIVE CSO - CHICAGO BOARD OF TRADE",
        "CORN CSO - CHICAGO BOARD OF TRADE",
    ],

    "Oilseeds": [
        "SOYBEANS - CHICAGO BOARD OF TRADE",
        "MINI SOYBEANS - CHICAGO BOARD OF TRADE",
        "SOYBEAN CSO - CHICAGO BOARD OF TRADE",
        "SOYBEAN OIL - CHICAGO BOARD OF TRADE",
        "SOYBEAN MEAL - CHICAGO BOARD OF TRADE",
        "USD Malaysian Crude Palm Oil C - CHICAGO MERCANTILE EXCHANGE",
        "CANOLA - ICE FUTURES U.S.",
    ],

    "Softs": [
        "COTTON NO. 2 - ICE FUTURES U.S.",
        "COCOA - ICE FUTURES U.S.",
        "SUGAR NO. 11 - ICE FUTURES U.S.",
        "COFFEE C - ICE FUTURES U.S.",
        "COFFEE CALENDAR SPREAD OPTIONS - ICE FUTURES U.S.",
    ],
}


CFTC Report DataFrame

CFTC commercial / non- commerical players

In [158]:

field_groups = {
    "Open Interest": [
        c for c in cftc_df.columns
        if c.startswith("Open_Interest")
    ],

    "Managed Money": [
        c for c in cftc_df.columns
        if c.startswith("M_Money_")
    ],

    "Producer Merchant": [
        c for c in cftc_df.columns
        if c.startswith("Prod_Merc_")
    ],

    "Swap Dealers": [
        c for c in cftc_df.columns
        if c.startswith("Swap_")
    ],

    "Other Reportable": [
        c for c in cftc_df.columns
        if c.startswith("Other_Rept_")
    ],

    "Non-Reportable": [
        c for c in cftc_df.columns
        if c.startswith("NonRept_")
    ],
}

CFTC commercial/non commerical positioning

In [159]:
position_groups = [
    "Managed Money",
    "Producer Merchant",
    "Swap Dealers",
    "Other Reportable",
    "Non-Reportable",  
]

#net positions of commerical / non commercial groups
net_columns = {}

for group in position_groups:

    columns = field_groups[group]

    long_col = next(
        (c for c in columns if "Long_All" in c),
        None
    )

    short_col = next(
        (c for c in columns if "Short_All" in c),
        None
    )

    if long_col and short_col:

        net_name = group.replace(" ", "_") + "_Net"

        net_columns[net_name] = (
            cftc_df[long_col] - cftc_df[short_col]
        )

# Add all net columns at once
cftc_df = pd.concat(
    [cftc_df, pd.DataFrame(net_columns, index=cftc_df.index)],
    axis=1
)

Most Recent Report View

In [160]:

latest_date = cftc_df["Date"].max()

date_filtered = cftc_df[
    (cftc_df["Date"] == latest_date) &
    (cftc_df["Market_and_Exchange_Names"].isin(market_categories["Grains"]))
]

columns = ["Report_Date_as_YYYY-MM-DD",
            "Market_and_Exchange_Names"]

for group in position_groups:
    fields = field_groups[group]

    for field in fields:
        if "Long_All" in field or "Short_All" in field:
            columns.append(field)

    columns.append(
        group.replace(" ", "_") + "_Net"
    )

date_filtered[columns]

,Report_Date_as_YYYY-MM-DD,Market_and_Exchange_Names,M_Money_Positions_Long_All,M_Money_Positions_Short_All,Managed_Money_Net,Prod_Merc_Positions_Long_All,Prod_Merc_Positions_Short_All,Producer_Merchant_Net,Swap_Positions_Long_All,Swap__Positions_Short_All,Swap_Dealers_Net,Other_Rept_Positions_Long_All,Other_Rept_Positions_Short_All,Other_Reportable_Net,NonRept_Positions_Long_All,NonRept_Positions_Short_All,Non-Reportable_Net
96030,2026-09-22,WHEAT-SRW - CHICAGO BOARD OF TRADE,85982,99126,-13144,79470,142485,-63015,89952,19625,70327,26754,26177,577,40413,35158,5255
96031,2026-09-22,WHEAT-HRW - CHICAGO BOARD OF TRADE,77023,35280,41743,22478,129396,-106918,82564,3434,79130,7619,22809,-15190,22582,21346,1236
96032,2026-09-22,WHEAT-HRSpring - MIAX FUTURES EXCHANGE,24562,3165,21397,32322,64550,-32228,6637,0,6637,1775,867,908,10820,7534,3286
96033,2026-09-22,CORN - CHICAGO BOARD OF TRADE,477679,63242,414437,478764,1204851,-726087,364397,43739,320658,101451,56120,45331,193559,247898,-54339
96034,2026-09-22,CORN CSO - CHICAGO BOARD OF TRADE,220,1964,-1744,8015,4952,3063,27,0,27,952,3172,-2220,1593,720,873


Seasonality

In [161]:

seasonality_columns = [
    "Date",
    "Market_and_Exchange_Names"
]

for group in position_groups:
    fields = field_groups[group]

    for field in fields:
        if "Long_All" in field or "Short_All" in field:
            seasonality_columns.append(field)

    seasonality_columns.append(
        group.replace(" ", "_") + "_Net"
    )

seasonality_filtered = cftc_df[
    cftc_df["Market_and_Exchange_Names"].isin(
        market_categories["Grains"]
    )
][seasonality_columns].copy()

seasonality_filtered

,Date,Market_and_Exchange_Names,M_Money_Positions_Long_All,M_Money_Positions_Short_All,Managed_Money_Net,Prod_Merc_Positions_Long_All,Prod_Merc_Positions_Short_All,Producer_Merchant_Net,Swap_Positions_Long_All,Swap__Positions_Short_All,Swap_Dealers_Net,Other_Rept_Positions_Long_All,Other_Rept_Positions_Short_All,Other_Reportable_Net,NonRept_Positions_Long_All,NonRept_Positions_Short_All,Non-Reportable_Net
0,2019-01-08,WHEAT-SRW - CHICAGO BOARD OF TRADE,94047,110526,-16479,71015,131497,-60482,84382,10716,73666,34110,22251,11859,38442,47006,-8564
1,2019-01-08,BLACK SEA WHEAT FINANCIAL - CHICAGO BOARD OF T...,2642,0,2642,7592,11210,-3618,1500,0,1500,276,500,-224,390,690,-300
2,2019-01-08,WHEAT-HRW - CHICAGO BOARD OF TRADE,69307,72991,-3684,72989,124203,-51214,46951,7484,39467,15769,9703,6066,30988,21622,9366
3,2019-01-08,WHEAT-HRSpring - MINNEAPOLIS GRAIN EXCHANGE,5030,15743,-10713,33344,24605,8739,2151,0,2151,5055,6115,-1060,12773,11891,882
4,2019-01-08,CORN - CHICAGO BOARD OF TRADE,265315,190121,75194,494637,803796,-309159,205116,30155,174961,110293,49703,60590,219776,221361,-1585
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96030,2026-09-22,WHEAT-SRW - CHICAGO BOARD OF TRADE,85982,99126,-13144,79470,142485,-63015,89952,19625,70327,26754,26177,577,40413,35158,5255
96031,2026-09-22,WHEAT-HRW - CHICAGO BOARD OF TRADE,77023,35280,41743,22478,129396,-106918,82564,3434,79130,7619,22809,-15190,22582,21346,1236
96032,2026-09-22,WHEAT-HRSpring - MIAX FUTURES EXCHANGE,24562,3165,21397,32322,64550,-32228,6637,0,6637,1775,867,908,10820,7534,3286
96033,2026-09-22,CORN - CHICAGO BOARD OF TRADE,477679,63242,414437,478764,1204851,-726087,364397,43739,320658,101451,56120,45331,193559,247898,-54339


Test


In [162]:

plot_df = seasonality_filtered.copy()


In [163]:
# Filters

markets = sorted(
    plot_df["Market_and_Exchange_Names"]
    .dropna()
    .unique()
)

market_dropdown = widgets.Dropdown(
    options=markets,
    value=markets[0],
    description="Market:",
    layout=widgets.Layout(width="600px")
)

group_dropdown = widgets.Dropdown(
    options=position_groups,
    value=position_groups[0],
    description="Group:",
    layout=widgets.Layout(width="350px")
)

type_dropdown = widgets.Dropdown(
    options=["Net", "Long", "Short"],
    value="Net",
    description="Position:",
    layout=widgets.Layout(width="250px")
)

In [164]:
def get_position_column(group, position_type):

    fields = field_groups[group]

    if position_type == "Net":
        column = group.replace(" ", "_") + "_Net"

    elif position_type == "Long":
        column = next(
            field for field in fields
            if "Long_All" in field
        )

    else:
        column = next(
            field for field in fields
            if "Short_All" in field
        )

    if column not in plot_df.columns:
        raise KeyError(f"Column not found: {column}")

    return column

In [176]:
def plot_recent(
    market,
    group,
    position_type,
    lookback
):

    value_column = get_position_column(
        group,
        position_type
    )

    market_df = plot_df[
        plot_df["Market_and_Exchange_Names"] == market
    ].copy()

    # Make sure Date is datetime
    market_df["Date"] = pd.to_datetime(
        market_df["Date"],
        errors="coerce"
    )

    # Make position column numeric
    market_df[value_column] = pd.to_numeric(
        market_df[value_column],
        errors="coerce"
    )

    # Latest available date
    latest_date = market_df["Date"].max()

    # Lookback period
    months = {
        "3M": 3,
        "6M": 6,
        "1Y": 12
    }[lookback]

    start_date = latest_date - pd.DateOffset(months=months)

    # Filter recent data
    recent_df = market_df[
        (market_df["Date"] >= start_date) &
        (market_df["Date"] <= latest_date)
    ].dropna(
        subset=["Date", value_column]
    )

    # Sort by date
    recent_df = recent_df.sort_values("Date")

    fig = px.bar(
        recent_df,
        x="Date",
        y=value_column,
        title=f"{market} — {group} {position_type}",
        labels={
            "Date": "Report Date",
            value_column: "Contracts"
        }
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="gray"
    )

    fig.update_layout(
        template="plotly_white",
        hovermode="x unified",
        height=500
    )

    return fig

In [177]:
recent_output = widgets.Output()

def update_recent(*args):
    with recent_output:
        recent_output.clear_output(wait=True)

        fig = plot_recent(
            market_dropdown.value,
            group_dropdown.value,
            type_dropdown.value,
            lookback_dropdown.value
        )

        display(fig)


market_dropdown.observe(update_recent, names="value")
group_dropdown.observe(update_recent, names="value")
type_dropdown.observe(update_recent, names="value")
lookback_dropdown.observe(update_recent, names="value")

display(
    market_dropdown,
    group_dropdown,
    type_dropdown,
    lookback_dropdown,
    recent_output
)

update_recent()

Dropdown(description='Market:', index=7, layout=Layout(width='600px'), options=('BLACK SEA WHEAT FINANCIAL - C…

Dropdown(description='Group:', layout=Layout(width='350px'), options=('Managed Money', 'Producer Merchant', 'S…

Dropdown(description='Position:', layout=Layout(width='250px'), options=('Net', 'Long', 'Short'), value='Net')

ToggleButtons(description='Lookback:', options=('3M', '6M', '1Y'), value='3M')

Output()

In [183]:
def plot_seasonality(
    market,
    group,
    position_type

):

    value_column = get_position_column(
        group,
        position_type
    )

    # Filter selected market
    season_df = plot_df[
        plot_df["Market_and_Exchange_Names"] == market
    ].copy()

    # Prepare date and position columns
    season_df["Date"] = pd.to_datetime(
        season_df["Date"],
        errors="coerce"
    )

    season_df[value_column] = pd.to_numeric(
        season_df[value_column],
        errors="coerce"
    )

    # Remove invalid observations
    season_df = season_df.dropna(
        subset=["Date", value_column]
    )

    # Use ISO year + ISO week together
    iso = season_df["Date"].dt.isocalendar()

    season_df["Year"] = iso.year.astype(int)
    season_df["Week"] = iso.week.astype(int)

    # Sort chronologically
    season_df = season_df.sort_values("Date")

    # Plot one line per year
    fig = px.line(
        season_df,
        x="Week",
        y=value_column,
        color="Year",
        title=f"{market} — {group} {position_type} Seasonality",
        labels={
            "Week": "Week of Year",
            value_column: "Contracts",
            "Year": "Report Year"
        },
        color_discrete_sequence=px.colors.qualitative.Set2
    )

    # Zero reference line
    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="gray"
    )

    fig.update_layout(
        template="plotly_white",
        hovermode="x unified",
        height=550,
        xaxis=dict(
            tickmode="linear",
            dtick=4,
            range=[1, 53]
        )
    )

    return fig

In [184]:
def update_seasonality(market, group, position_type):

    fig = plot_seasonality(
        market,
        group,
        position_type
    )

    display(fig)


season_output = widgets.interactive_output(
    update_seasonality,
    {
        "market": market_dropdown,
        "group": group_dropdown,
        "position_type": type_dropdown
    }
)

display(
    market_dropdown,
    group_dropdown,
    type_dropdown,
    season_output
)

Dropdown(description='Market:', index=1, layout=Layout(width='600px'), options=('BLACK SEA WHEAT FINANCIAL - C…

Dropdown(description='Group:', layout=Layout(width='350px'), options=('Managed Money', 'Producer Merchant', 'S…

Dropdown(description='Position:', layout=Layout(width='250px'), options=('Net', 'Long', 'Short'), value='Net')

Output()

In [190]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display


# ============================================================
# GET AVAILABLE YEARS FOR SELECTED MARKET
# ============================================================

def get_available_years(market):

    temp = plot_df[
        plot_df["Market_and_Exchange_Names"] == market
    ].copy()

    temp["Date"] = pd.to_datetime(
        temp["Date"],
        errors="coerce"
    )

    temp = temp.dropna(
        subset=["Date"]
    )

    iso = temp["Date"].dt.isocalendar()

    years = sorted(
        iso.year.dropna().astype(int).unique().tolist()
    )

    return years


# ============================================================
# YEAR SELECTOR
# ============================================================

initial_years = get_available_years(
    market_dropdown.value
)

year_dropdown = widgets.SelectMultiple(
    options=["All"] + initial_years,
    value=("All",),
    description="Years:",
    rows=8,
    layout=widgets.Layout(
        width="250px",
        height="180px"
    )
)


# ============================================================
# SEASONALITY PLOT
# ============================================================

def plot_seasonality(
    market,
    group,
    position_type,
    selected_years=("All",)
):

    value_column = get_position_column(
        group,
        position_type
    )

    # --------------------------------------------------------
    # Filter selected market
    # --------------------------------------------------------

    season_df = plot_df[
        plot_df["Market_and_Exchange_Names"] == market
    ].copy()

    # --------------------------------------------------------
    # Prepare date and position columns
    # --------------------------------------------------------

    season_df["Date"] = pd.to_datetime(
        season_df["Date"],
        errors="coerce"
    )

    season_df[value_column] = pd.to_numeric(
        season_df[value_column],
        errors="coerce"
    )

    # Remove invalid observations
    season_df = season_df.dropna(
        subset=["Date", value_column]
    )

    # --------------------------------------------------------
    # ISO year + week
    # --------------------------------------------------------

    iso = season_df["Date"].dt.isocalendar()

    season_df["Year"] = (
        iso.year.astype(int)
    )

    season_df["Week"] = (
        iso.week.astype(int)
    )

    # --------------------------------------------------------
    # YEAR FILTER
    # --------------------------------------------------------

    if "All" not in selected_years:

        season_df = season_df[
            season_df["Year"].isin(
                selected_years
            )
        ]

    # --------------------------------------------------------
    # Sort
    # --------------------------------------------------------

    season_df = season_df.sort_values(
        "Date"
    )

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    fig = px.line(
        season_df,
        x="Week",
        y=value_column,
        color="Year",

        title=(
            f"{market} — "
            f"{group} "
            f"{position_type} "
            f"Seasonality"
        ),

        labels={
            "Week": "Week of Year",
            value_column: "Contracts",
            "Year": "Report Year"
        },

        color_discrete_sequence=(
            px.colors.qualitative.Set2
        )
    )

    # Zero reference line
    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="gray"
    )

    # Layout
    fig.update_layout(
        template="plotly_white",
        hovermode="x unified",
        height=550,

        xaxis=dict(
            tickmode="linear",
            dtick=4,
            range=[1, 53]
        )
    )

    return fig


# ============================================================
# UPDATE YEAR OPTIONS WHEN MARKET CHANGES
# ============================================================

def update_year_options(change):

    if change["name"] != "value":
        return

    market = change["new"]

    years = get_available_years(
        market
    )

    # Update available years
    year_dropdown.options = (
        ["All"] + years
    )

    # Reset to All
    year_dropdown.value = (
        "All",
    )


market_dropdown.observe(
    update_year_options,
    names="value"
)


# ============================================================
# HANDLE YEAR SELECTION
# ============================================================

def update_year_selection(change):

    selected = change["new"]

    # Nothing selected
    if len(selected) == 0:

        year_dropdown.value = (
            "All",
        )

        return

    # If All + another year are selected,
    # remove All
    if (
        "All" in selected
        and len(selected) > 1
    ):

        year_dropdown.value = tuple(
            year
            for year in selected
            if year != "All"
        )


year_dropdown.observe(
    update_year_selection,
    names="value"
)


# ============================================================
# UPDATE SEASONALITY
# ============================================================

def update_seasonality(
    market,
    group,
    position_type,
    selected_years
):

    fig = plot_seasonality(
        market,
        group,
        position_type,
        selected_years
    )

    display(fig)


# ============================================================
# INTERACTIVE OUTPUT
# ============================================================

season_output = widgets.interactive_output(
    update_seasonality,
    {
        "market": market_dropdown,
        "group": group_dropdown,
        "position_type": type_dropdown,
        "selected_years": year_dropdown
    }
)


# ============================================================
# DISPLAY
# ============================================================

display(
    market_dropdown,
    group_dropdown,
    type_dropdown,
    year_dropdown,
    season_output
)

Dropdown(description='Market:', index=1, layout=Layout(width='600px'), options=('BLACK SEA WHEAT FINANCIAL - C…

Dropdown(description='Group:', layout=Layout(width='350px'), options=('Managed Money', 'Producer Merchant', 'S…

Dropdown(description='Position:', layout=Layout(width='250px'), options=('Net', 'Long', 'Short'), value='Net')

SelectMultiple(description='Years:', index=(0,), layout=Layout(height='180px', width='250px'), options=('All',…

Output()